In [1]:

import pandas as pd
import json
from Jumpit_Crawling_Functions import *
from Data_Analysis import *



In [2]:

mergeData = pd.read_json('../Data_Files/Crawling_DataFile_MergePage_json_20250512.txt')

mergeDataTrans = mergeData.T
mergeDataTrans

# jobCategory의 리스트 값들을 원핫인코딩 해서 mergeDataTrans 칼럼에 넣는 방식

jobCategory = mergeDataTrans['jobCategory'].apply(lambda x : x.split(','))
jobCategory = jobCategory.explode()
jobCategory = pd.get_dummies(jobCategory)


education = mergeDataTrans['education']

educationJobCategory = combineDataFrames(jobCategory, education)
educationJobCategory

#education별로 jobCategory들의 개수 세기
educationJobCategory = educationJobCategory.groupby('education').sum()
educationJobCategory

dataDict = educationJobCategory.to_dict()
dataDict

wrappedDict = {
    '직무별 요구하는 학력': dataDict
}

file_path = '../Data_Files/Data_Analysis.txt'
originalJson = getOriginalJson(file_path)
mergeJson = mergeJsonDicts(originalJson, wrappedDict)






In [3]:
saveData(mergeJson, file_path)

새로운 JSON 파일을 생성했습니다.


In [ ]:
dataCleaned = data.drop(columns=['title_y','companyName','techStack_x','jobLocation_y','date_y','career'],
          axis=1, inplace=False)

In [ ]:
dataCleaned.rename({'title_x': 'title', 'jobLocation_x': 'jobLocation','techStack_y':'techStack'},
                   axis=1, inplace=True)
dataRenamed = dataCleaned.rename(
    {
        'title':'제목',
        'company' : '회사명',
        'pride' : '특이사항',
        'mainWork' : '주요업무',
        'requirements' : '자격요건',
        'preferential' : '우대사항',
        'welfare' : '복지 및 혜a택',
        'procedures' : '채용 절차',
        'date_x' : '공고일',
        'jobCategory' : '직무',
        'minCareer' : '최소 경력',
        'maxCareer' : '최대 경력',
        'techStack' : '기술 스택'
    }
    ,inplace=False
)

dataRenamed.info()

In [ ]:
dataCleaned

In [ ]:
categoryEducation = data[['title_x', 'jobCategory', 'education']]



In [ ]:
categoryEducation = categoryEducation.rename(
    columns={'title_x': 'title', 'jobCategory': 'category', 'education': 'education'})
categoryEducation = categoryEducation.drop_duplicates(subset=['title', 'category', 'education'], keep='first')
categoryEducation = categoryEducation.reset_index(drop=True)
categoryEducation = categoryEducation.dropna()


In [ ]:
categoryEducation['category'].info()

In [ ]:
categoryEducation.category.value_counts()



In [ ]:
techStackList = data['techStack_x'].to_list()
techStackList
techStacks = sum(techStackList,[])
techStacks

In [ ]:
data2 = pd.DataFrame(techStacks)
data2.value_counts()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib as mpl
from matplotlib import rc

# 한글 폰트 설정
# font_path = 'C:/Windows/Fonts/malgun.ttf'  # Windows의 경우
# font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'  # Linux의 경우
#font_path = 'System/Library/Fonts/AppleGothic.ttf'  # Mac의 경우
#font_prop = fm.FontProperties(fname=font_path, size=12)
#mpl.rc('font', family=font_prop.get_name())
#mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
rc('font', family='AppleGothic')  ## 이 두 줄을
plt.rcParams['axes.unicode_minus'] = False  ## 추가해줍니다. 

# 그래프 크기 설정
plt.figure(figsize=(10, 6))

# 그래프 스타일 설정
sns.set_style("whitegrid")

# 그래프 그리기
sns.barplot(x=techStackTop10['TechStack'], y=techStackTop10['Count'], palette='viridis')
plt.title('Top 10 Tech Stacks', fontsize=16, fontproperties=font_prop)
plt.xlabel('Tech Stack', fontsize=14, fontproperties=font_prop)
plt.ylabel('Count', fontsize=14, fontproperties=font_prop)
plt.xticks(rotation=45, fontsize=12, fontproperties=font_prop)
plt.yticks(fontsize=12, fontproperties=font_prop)
plt.show()




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# 그래프 크기 설정
plt.figure(figsize=(10, 6))

# 그래프 스타일 설정
sns.set_style("whitegrid")

# 그래프 그리기
# (여기서는 techStackTop10 DataFrame이 정의되어 있다고 가정합니다.)
sns.barplot(x=techStackTop10['TechStack'], y=techStackTop10['Count'], palette='viridis')
plt.title('Top 10 Tech Stacks', fontsize=16)
plt.xlabel('Tech Stack', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.show()

In [ ]:
# techStack 안의 리스트를 전부 분해해서 새로운 행으로 만들기
data0504Exploded = data0504.explode('techStack')
data0504Exploded.reset_index(drop=True, inplace=True)
data0504Exploded


In [ ]:
techCareerData = data0504Exploded[['minCareer', 'techStack']]

In [ ]:
techStackTop10List = techStackTop10['TechStack'].tolist()
techStackTop10List

In [ ]:
top10Data = techCareerData[techCareerData['techStack'].isin(techStackTop10List)].copy()
top10Data

In [ ]:
top10Data.index = top10Data['techStack']
top10Data = top10Data.drop(columns=['techStack'])


In [ ]:
# 테크 스택별로 보통 몇년의 경력을 요구하는지 보여주는 그래프 생성
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")
# viridis 말고 다른 팔레트도 사용해보세요. 예: 'coolwarm', 'plasma', 'magma', 'cividis'
sns.barplot(x=top10Data.index, y=top10Data['minCareer'], palette='plasma')
plt.title('Minimum Career Requirement by Tech Stack', fontsize=16, fontproperties=font_prop)
plt.xlabel('Tech Stack', fontsize=14, fontproperties=font_prop)
plt.ylabel('Minimum Career (Years)', fontsize=14, fontproperties=font_prop)
plt.xticks(rotation=45, fontsize=12, fontproperties=font_prop)
plt.yticks(fontsize=12, fontproperties=font_prop)
plt.show()



In [ ]:
# 탑 10 기술스택 별로 평균 계산

techStackTop10Avg = top10Data.groupby('techStack')['minCareer'].mean().reset_index()
techStackTop10Avg.columns = ['TechStack', 'AvgMinCareer']
techStackTop10Avg = techStackTop10Avg.sort_values(by='AvgMinCareer', ascending=False)
techStackTop10Avg


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# mac 한글 폰트 설정
font_path = '/Library/Fonts/Arial Unicode.ttf'  # Mac 시스템 기본 고딕 폰트 경로
font_prop = fm.FontProperties(fname=font_path, size=12)
mpl.rc('font', family=font_prop.get_name())
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 그래프 크기 설정

plt.figure(figsize=(12, 8))

sns.set_style("whitegrid")
# viridis 말고 다른 팔레트도 사용해보세요. 예: 'coolwarm', 'plasma', 'magma', 'cividis'
sns.barplot(x=techStackTop10Avg['AvgMinCareer'], y=techStackTop10Avg['TechStack'], palette='coolwarm', ci=None)
plt.title('기술 스택별 요구되는 평균 경럭', fontsize=16, fontproperties=font_prop)
plt.xlabel('평균 경력 (년)', fontsize=14, fontproperties=font_prop)
plt.ylabel('기술 스택', fontsize=14, fontproperties=font_prop)
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)
plt.show()

In [ ]:
data_detail

In [ ]:
import pandas as pd
dataYesterday = pd.read_json('../Data_Files/Crawling_DataFile_DetailPage_json_20250510.txt')
dataToday = pd.read_json('../Data_Files/Crawling_DataFile_DetailPage_json_20250511.txt')

In [ ]:
dataYesterday

In [ ]:
dataToday

In [ ]:
dataNew = dataToday[~dataToday.index.isin(dataYesterday.index)]

In [ ]:
dataNew

In [ ]:
dataClosed = dataYesterday[~dataYesterday.index.isin(dataToday.index)]

In [ ]:
dataClosed = getClosedPost(dataYesterday, dataToday)
dataClosed

In [ ]:
# 특정 techStack의 지역별 분포

stackLocation = dataToday[['techStack', 'jobLocation']]
stackLocation

In [ ]:
stackLocation = stackLocation.explode('techStack')
stackLocation
